# Briefing
Esse notebook tem como objetivo treinar um modelo de classificação de eventos sonoros. Utilizou o dataset **UrbanSound8K**, pré-separado em 10 folds, com 10 categorias de eventos sonoros.

Utilizou-se um Multilayer Perceptron para classificar o áudio com base no seu MFCC, um tipo de espectrograma específico capaz de comprimir a informação tempo-frequência em poucos `bins` espectrais.

# Importando tudo

In [ ]:
# !!Panic Button
# import os
# os._exit(0)

In [2]:
from IPython.display import clear_output
# !pip install --upgrade ipython
# %load_ext autoreload
# %autoreload 2

!git clone https://github.com/y7kko/COM804-Classifier.git ykk_repo
!git -C ./ykk_repo/ pull
!pip install git+https://github.com/soundata/soundata.git
clear_output()
import sys
import os

sys.path.append('./ykk_repo/')
os.environ['YKK_NOCHDIR'] = '1'
os.environ['US8K_PATH'] = './data/urbansound8k'
import ykk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import optuna
ykk.urban8k(root_path=os.environ['US8K_PATH']).download_dataset()

print('Done, check console for more details')

Done, check console for more details


# Declarando

## Dataset importer

In [26]:
def get_data(val_fold,fs,n_mfcc=13,mel_kw=None,return_dataloader=True):
    """Splits dataset into validation fold and training fold."""
    train_fld = [f for f in range(1, 11) if f != val_fold]
    valid_data = ykk.urban8k(augmentation=False, fs=fs, filter_folds=val_fold)
    train_data = ykk.urban8k(augmentation=True, fs=fs, filter_folds=train_fld)

    if not (mel_kw is None):
        for _data in [train_data,valid_data]:
            _data.set_mfcc_params(n_mfcc = n_mfcc, mel_kw = mel_kw)

    if return_dataloader:
        train_dl = DataLoader(train_data,shuffle=True,batch_size=100,
                              num_workers=3
                             )
        valid_dl = DataLoader(valid_data,batch_size=200)
        return train_dl,valid_dl
    else:
        return train_data,valid_data

## Parâmetros
- `fs`: Frequência de amostragem do sinal
- `mel_kw`: Parâmetros do espectrograma
- `n_mfcc`: número de componentes de frequência por frame de tempo

Nota: para simplificar, o tamanho da input layer foi calculado realizando a extração de features, mas há forma analítica de calcular.

In [27]:
fs = 44100
mel_kw = {"n_fft": 2048, "hop_length": 2048, "n_mels": 40}
n_mfcc = 20

_d = ykk.urban8k(root_path=os.environ['US8K_PATH'],fs=fs)
_d.set_mfcc_params(n_mfcc = n_mfcc, mel_kw = mel_kw)
input_shape = list((_d[0][0]).shape)
print(f'input shape: {input_shape} ({np.prod(input_shape)})')

input shape: [1, 20, 87] (1740)


## O modelo
O modelo conta com duas hidden layers de perceptrons. A padronização é realizada com `BatchNorm1d`.

Uma boa alternativa para o MLP, em tarefas relacionadas à áudio, é utilizar camadas convolucionais, visto que um espectrograma é essencialmente uma imagem.

In [28]:
class NeuralNetwork(nn.Module):
    def __init__(self,dropout=.5):
        super().__init__()
        # Nx-|K|+1 = 20 - 5
        # Ny-|K|+1 = 173 - 5
        self.sequence = nn.Sequential(
            # Layer1 conv
            nn.Conv2d(1, 16, kernel_size=5),   # (B, 16, 16, 83)
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),      # !(B, 16, 8, 41)
            
            # Layer2 conv
            nn.Conv2d(16, 32, kernel_size=3), # (B, 32, 6, 39)
            nn.ReLU(inplace=True),

            # Pooling
            nn.MaxPool2d(2),                  # !(B, 32, 3, 19)
            nn.Flatten(),                      # (B, 1824)
            
            # Layer 3 Lin
            nn.Linear(32 * 3 * 19, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # Layer 4 lin
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout),

            # Layer 5
            nn.Linear(32, 10),
        )


    def forward(self, x):
        x = self.sequence(x)
        return x

n_params = sum(p.numel() for p in NeuralNetwork().parameters())
print(f'Number o parameters {n_params/1E3}E3')

Number o parameters 124.266E3


## Trainer


In [48]:
from sklearn.metrics import f1_score
class Trainer():
    def __init__(self,model,loss_fn,valid_loss_delta=1E-4):
        self.device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
        print(f'device: {self.device}')
        self.model = model.to(self.device)
        self.loss_fn = loss_fn
        self.valid_loss = []
        self.train_loss = []
        self.accuracy = []
        self.f1_score = []


    def train(self,train_data:DataLoader, optimizer):
        size = len(train_data.dataset)
        self.model.train()

        running_loss,seen = 0.0, 0
        for batch, (X, y) in enumerate(train_data):
            X, y = self._dev_encode(X,y)
            # Compute prediction error
            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            # Backpropagation
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            running_loss += loss.item()*len(X)
            seen += len(X)
            if batch % 50 == 0:
                loss, current = loss.item(), (batch + 1) * len(X)
                print(f"\t loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")
        self.train_loss.append(running_loss/seen)
        return running_loss/seen

    
    @torch.no_grad()
    def validate(self, test_data):
        size = len(test_data.dataset)
        self.model.eval()
        valid_loss, correct = 0.0, 0
    
        all_preds, all_labels = [], []
        for X, y in test_data:
            X, y = self._dev_encode(X, y)
    
            logits = self.model(X)                       # logits crus
            valid_loss += self.loss_fn(logits, y).item() * len(X)
            preds = logits.argmax(1)
            correct += (preds == y).sum().item()
    
            all_preds.append(preds.cpu())
            all_labels.append(y.cpu())
    
        valid_loss /= size
        correct /= size
    
        all_preds  = torch.cat(all_preds).numpy()
        all_labels = torch.cat(all_labels).numpy()
        f1 = f1_score(all_labels, all_preds, average='macro')
    
        self.valid_loss.append(valid_loss)
        self.accuracy.append(correct * 100)
        self.f1_score.append(f1)
    
        print(f"\t Accuracy: {100*correct:>0.1f}%, Avg loss: {valid_loss:>8f}, f1: {f1:.4f}\n")
        return valid_loss, correct, f1
    
    def _dev_encode(self,*args):
        return tuple(arg.to(self.device) for arg in args)


## Treino de um fold
Para cada fold, é chamado `evaluate_fold`, reponsável por separar o dataset em validação e teste. Alguns hiperparâmetros foram expostos para otimização com optuna, sendo estes:
- dropout
- learning rate
- dimensões das hidden layers

In [49]:
import time
import copy
def evaluate_fold(fold,dropout,lr,max_epochs=50,patience=7,):
    train_dl,valid_dl = get_data(fold, fs=fs, n_mfcc = n_mfcc, mel_kw = mel_kw)

    model = NeuralNetwork(dropout=dropout)
    # model.compile()
    loss_fn = nn.CrossEntropyLoss()
    # optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    trainer = Trainer(model = model, loss_fn = loss_fn)
    best_state = None
    best_correct, early_stop_cnt, best_val_loss,best_f1 = 0, 0, float('inf'),0
    for epoch in range(max_epochs):
        print(f':: ({fold}) Epoch: {epoch}')
        inicio_total = time.time()
        trainer.train(train_dl, optimizer = optimizer)
        val_loss, correct,f1 = trainer.validate(valid_dl)
    
        # EarlyStopper
        if f1 > best_f1 + 1e-3:
            best_f1 = f1
            best_val_loss = val_loss
            best_correct = correct
            early_stop_cnt = 0
            best_state = copy.deepcopy(trainer.model.state_dict())
        else:
            early_stop_cnt += 1
            if early_stop_cnt >= patience:
                break

        duracao = time.time() - inicio_total
        print(f"\t{duracao/60:.2f} min ({duracao/3600:.2f} h)")
    trainer.model.load_state_dict(best_state)
    return best_val_loss, best_f1,best_model

In [ ]:
coisa = evaluate_fold(1,.5,1E-2,max_epochs=50,patience=7,)

# Otimização de parâmetros
utilizando o optuna para encontrar um bom valor para learning rate e dropout.

~~só deus sabe o que eu fiz com esses prints de loss de acordo com o batch~~

In [ ]:
def objective(trial):
    dropout = trial.suggest_float("dropout", 0.0, 0.5)
    lr = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    val_losses, val_f1 = [], []
    # Cada epoch dura ~10 min, 
    # testar todos os folds dá timeout no kaggle, selecionei alguns folds aleatoriamente
    for fold in [1,3,9]:
        vl,vf1, _ = evaluate_fold(fold, dropout=dropout, lr=lr, max_epochs=5, patience = 7)
        val_losses.append(vl)
        val_f1.append(vf1)
        trial.report(np.mean(val_f1), fold)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(val_f1))
    
study_name = "dropout_lr_study_1"
storage_name = f"sqlite:///{study_name}.db"

study = optuna.create_study(
    study_name = study_name,
    storage    = storage_name,
    direction  = "maximize",
    sampler    = optuna.samplers.TPESampler(seed=42),
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, 
                                             n_warmup_steps=2),
    load_if_exists = True
)
study.optimize(objective, 
               n_trials=30, 
               timeout=None)
print("Best params:", study.best_params)
print("Best value:", study.best_value)

[I 2026-11-04 20:12:58,701] Using an existing study with `study_name='dropout_lr_study_1'` instead of creating a new one.


device: cpu
:: (1) Epoch: 0
	 loss: 2.306194  [  100/ 7859]
